# Persistent Cell Tracking

## Tracking Strategy

- **Persistent cell IDs** — Each cell is assigned a unique global ID that persists throughout its entire lifecycle
- **Centroid-based tracking** — Cells are linked between frames using centroid displacement
- **Hungarian algorithm** — Optimal one-to-one assignment of cells between consecutive frames
- **New cell detection** — Automatically detects newly appearing cells or division events
- **Data consistency** — Persistent IDs enable downstream Fourier and shape analysis

## Output Files

### Analysis Tables

- `consistent_cell_boundaries.csv` — Boundary coordinates with persistent cell IDs
- `consistent_shape_summary.csv` — Shape metrics with consistent tracking
- `cell_trajectories.csv` — Cell movement and trajectory statistics
- `detailed_tracking_info.csv` — Complete per-frame tracking information

### Visualization Files

- `consistently_tracked_stack.tif` — Segmentation stack with persistent cell IDs
- `consistently_tracked_stack_rgb.tif` — Color-coded visualization
- `consistently_tracked_stack_with_ids.tif` — RGB stack with cell ID overlays

## User-Adjustable Parameters

- `max_distance = 50` — Maximum centroid displacement per frame (pixels)
- `min_area = 50` — Minimum cell area to be tracked (pixels)
- `max_gap = 5` — Frames a cell may be missing before termination

## Notes

- Tracking is performed in pixel space for numerical stability
- With a scale of 3.2184 pixels/µm, `max_distance = 50` pixels corresponds to ~15.5 µm

In [1]:
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path
from skimage.measure import regionprops, find_contours
import matplotlib.pyplot as plt
import cv2
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

#-------------------------------------------------------------------------------------------
# USER INPUT: pixel size from the orginal data
# Tracking parameters
# Scale: 3.2184 pixels / micron
# max_distance = 50 pixels ≈ 15.5 microns
#-------------------------------------------------------------------------------------------

PIXEL_SIZE_UM = 0.3107  # example: Scale = 3.2184 pixels / micron

# ---------------------------
# 1. TRACKING
# ---------------------------
def track_cells(stack_file, max_distance=50, min_area=50, max_gap=5): #<---change this, pixels
    
    mask_stack = tifffile.imread(stack_file)
    T = mask_stack.shape[0]

    tracked = np.zeros_like(mask_stack, dtype=np.uint32)

    next_id = 1
    active = {}
    records = []

    for t in range(T):
        frame = mask_stack[t]

        # --- extract cells ---
        current = {
            r.label: {'centroid': r.centroid, 'area': r.area}
            for r in regionprops(frame)
            if r.area >= min_area
        }

        if t == 0:
            for orig_id, cell in current.items():
                centroid = cell['centroid']

                active[next_id] = {'centroid': centroid, 'last_seen': t}
                tracked[t][frame == orig_id] = next_id

                records.append({
                    'timepoint': t,
                    'global_id': next_id,
                    'centroid_x': centroid[1],
                    'centroid_y': centroid[0],
                    'centroid_x_um': centroid[1] * PIXEL_SIZE_UM,
                    'centroid_y_um': centroid[0] * PIXEL_SIZE_UM,
                    'area': cell['area'],
                    'status': 'new'
                })

                next_id += 1
            continue

        assigned = set()

        if current and active:
            curr_ids = list(current.keys())
            curr_centroids = np.array([current[i]['centroid'] for i in curr_ids])

            act_ids = list(active.keys())
            act_centroids = np.array([active[i]['centroid'] for i in act_ids])

            D = cdist(curr_centroids, act_centroids)
            ci, ai = linear_sum_assignment(D)

            for c, a in zip(ci, ai):
                if D[c, a] <= max_distance:
                    orig_id = curr_ids[c]
                    gid = act_ids[a]

                    centroid = current[orig_id]['centroid']

                    active[gid] = {'centroid': centroid, 'last_seen': t}
                    tracked[t][frame == orig_id] = gid

                    records.append({
                        'timepoint': t,
                        'global_id': gid,
                        'centroid_x': centroid[1],
                        'centroid_y': centroid[0],
                        'centroid_x_um': centroid[1] * PIXEL_SIZE_UM,
                        'centroid_y_um': centroid[0] * PIXEL_SIZE_UM,
                        'area': current[orig_id]['area'],
                        'status': 'tracked'
                    })

                    assigned.add(orig_id)

        # --- new cells ---
        for orig_id, cell in current.items():
            if orig_id not in assigned:
                centroid = cell['centroid']

                active[next_id] = {'centroid': centroid, 'last_seen': t}
                tracked[t][frame == orig_id] = next_id

                records.append({
                    'timepoint': t,
                    'global_id': next_id,
                    'centroid_x': centroid[1],
                    'centroid_y': centroid[0],
                    'centroid_x_um': centroid[1] * PIXEL_SIZE_UM,
                    'centroid_y_um': centroid[0] * PIXEL_SIZE_UM,
                    'area': cell['area'],
                    'status': 'new'
                })

                next_id += 1

        # --- remove old tracks ---
        active = {
            gid: info for gid, info in active.items()
            if t - info['last_seen'] <= max_gap
        }

    tracking_df = pd.DataFrame(records)
    return tracked, tracking_df


# ---------------------------
# 2. LIFESPANS
# ---------------------------
def compute_lifespans(tracking_df):
    df = tracking_df.groupby('global_id')['timepoint'].agg(['min', 'max', 'count'])
    df['lifespan'] = df['max'] - df['min'] + 1
    return df


# ---------------------------
# 3. VISUALIZATION
# ---------------------------
def save_tracking_stack(tracked, output_folder):

    output = Path(output_folder)
    output.mkdir(exist_ok=True, parents=True)

    tifffile.imwrite(
        output / "tracked_stack.tif",
        tracked,
        metadata={'axes': 'TYX'}
    )


# ---------------------------
# 4. BOUNDARIES + SHAPE
# ---------------------------
def extract_boundaries(tracked, output_folder):

    rows = []

    for t in range(tracked.shape[0]):
        frame = tracked[t]

        for cid in np.unique(frame)[1:]:
            mask = frame == cid

            contours = find_contours(mask, 0.5)
            if not contours:
                continue

            boundary = max(contours, key=len)

            for i, (y, x) in enumerate(boundary):
                rows.append({
                    'time': t,
                    'cell_id': cid,
                    'point_id': i,
                    'x': x,
                    'y': y,
                    'x_um': x * PIXEL_SIZE_UM,
                    'y_um': y * PIXEL_SIZE_UM
                })

    df = pd.DataFrame(rows)
    df.to_csv(Path(output_folder) / "boundaries.csv", index=False)
    return df


# ---------------------------
# 5. TRAJECTORIES
# ---------------------------
def compute_trajectories(tracking_df, output_folder):

    rows = []

    for cid, df in tracking_df.groupby('global_id'):
        df = df.sort_values('timepoint')

        if len(df) < 5:
            continue

        coords = df[['centroid_x', 'centroid_y']].values
        d = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))

        rows.append({
            'cell_id': cid,
            'lifespan': len(df),
            'total_distance_um': (d * PIXEL_SIZE_UM).sum(),
            'avg_speed_um': (d * PIXEL_SIZE_UM).mean()
        })

    traj = pd.DataFrame(rows)
    traj.to_csv(Path(output_folder) / "trajectories.csv", index=False)
    return traj


# ---------------------------
# 6. PIPELINE
# ---------------------------
def run_pipeline(stack_file, output_folder):

    tracked, tracking_df = track_cells(stack_file)

    lifespans = compute_lifespans(tracking_df)

    save_tracking_stack(tracked, output_folder)
    extract_boundaries(tracked, output_folder)
    trajectories = compute_trajectories(tracking_df, output_folder)

    tracking_df.to_csv(Path(output_folder) / "tracking.csv", index=False)
    lifespans.to_csv(Path(output_folder) / "lifespans.csv")

    print(f"Tracked cells: {tracking_df['global_id'].nunique()}")

    return tracking_df, lifespans, trajectories

In [ ]:

# Run the complete pipeline on your data
results = run_pipeline(
    stack_file=r"masks_stack.tif",  #<----change this
    output_folder=r"output_folder", #<----change this
)

Tracked cells: 31


In [ ]:
# ---------------------------
# 00. QUICK VALIDATION
# ---------------------------
import pandas as pd
import numpy as np

# Load outputs
output_folder = r"output_folder", #<----change this
tracking_df   = pd.read_csv(f"{output_folder}/tracking.csv")
trajectories_df = pd.read_csv(f"{output_folder}/trajectories.csv")
boundaries_df = pd.read_csv(f"{output_folder}/boundaries.csv")

# 1️⃣ Check unique IDs
n_unique_cells = tracking_df['global_id'].nunique()
print(f"✅ Unique tracked cells: {n_unique_cells}")

# 2️⃣ Check pixel → µm conversion consistency
px_um_ratio = 0.3107  # Your PIXEL_SIZE_UM
centroid_check = tracking_df[['centroid_x', 'centroid_x_um', 'centroid_y', 'centroid_y_um']].head()
centroid_check['calc_x_um'] = centroid_check['centroid_x'] * px_um_ratio
centroid_check['calc_y_um'] = centroid_check['centroid_y'] * px_um_ratio

# Compare calculated µm to saved µm
centroid_check['x_diff'] = centroid_check['centroid_x_um'] - centroid_check['calc_x_um']
centroid_check['y_diff'] = centroid_check['centroid_y_um'] - centroid_check['calc_y_um']
print("\nSample centroid check (should be near 0):")
print(centroid_check[['centroid_x_um','calc_x_um','x_diff','centroid_y_um','calc_y_um','y_diff']])

# 3️⃣ Quick check on trajectories
tracking_ids = set(tracking_df['global_id'].unique())
traj_ids = set(trajectories_df['cell_id'].unique())
missing_traj = traj_ids - tracking_ids
if missing_traj:
    print(f"⚠️ Some trajectories cell IDs not in tracking_df: {missing_traj}")
else:
    print("✅ All trajectory cell IDs exist in tracking_df")

# 4️⃣ Quick check on boundaries
boundary_ids = set(boundaries_df['cell_id'].unique())
missing_boundaries = boundary_ids - tracking_ids
if missing_boundaries:
    print(f"⚠️ Some boundary cell IDs not in tracking_df: {missing_boundaries}")
else:
    print("✅ All boundary cell IDs exist in tracking_df")

✅ Unique tracked cells: 31

Sample centroid check (should be near 0):
   centroid_x_um   calc_x_um        x_diff  centroid_y_um  calc_y_um  \
0     228.478856  228.478856  0.000000e+00      28.940184  28.940184   
1     193.181875  193.181875  0.000000e+00      51.791587  51.791587   
2     155.507178  155.507178 -2.842171e-14      57.686932  57.686932   
3     113.695407  113.695407 -1.421085e-14      62.875142  62.875142   
4      75.454401   75.454401  0.000000e+00      90.523831  90.523831   

         y_diff  
0  3.552714e-15  
1  0.000000e+00  
2  0.000000e+00  
3  7.105427e-15  
4  0.000000e+00  
✅ All trajectory cell IDs exist in tracking_df
✅ All boundary cell IDs exist in tracking_df
